# Análisis estadístico batch F1_highrisk v11
Este notebook analiza las longitudes de episodio y métricas clave de los 10 runs del batch F1_highrisk (risk_scale=1.2), incluyendo intervalos de confianza, p-valores, tamaño del efecto y corrección de Bonferroni. Los resultados se exportan a la carpeta `analysis/` para trazabilidad y reporte.

## 1. Importar librerías y cargar datos
Importamos pandas, numpy, scipy.stats y matplotlib/seaborn. Cargamos los datos de longitudes de episodio desde los archivos JSON generados por los runs del batch.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import re

# Prefijos y rutas
seeds = [42, 101, 13, 7, 99]
grids = [8, 16]
# Ruta raíz del proyecto (ajustada manualmente)
project_root = Path(r'c:/Proyectos/TUI-v4.1')
base = project_root / 'results' / 'v11' / 'F1_highrisk' / 'raw'
files = [base / f'grid{g}_riskhigh_r1p2_seed{s}_v11_episodes.csv' for g in grids for s in seeds]

# Verificar existencia de archivos y cargar datos
missing = [str(f) for f in files if not f.exists()]
if missing:
    print('Archivos faltantes:', missing)

rows = []
for f in files:
    if not f.exists():
        continue
    df_csv = pd.read_csv(f)
    fname = f.name
    grid_match = re.search(r'grid(\d+)', fname)
    seed_match = re.search(r'seed(\d+)', fname)
    grid_val = int(grid_match.group(1)) if grid_match else None
    seed_val = int(seed_match.group(1)) if seed_match else None
    for agent in ['control','simbiosis','dqn_control']:
        df_agent = df_csv[df_csv['Agente'] == agent]
        lengths = df_agent.groupby('Episodio').size().values
        if len(lengths) == 0:
            continue
        rows.append({
            'grid': grid_val,
            'seed': seed_val,
            'agent': agent,
            'mean': np.mean(lengths),
            'std': np.std(lengths),
            'min': np.min(lengths),
            'max': np.max(lengths)
        })
df = pd.DataFrame(rows)
print('Carga completada. Registros:', len(df))

Carga completada. Registros: 30


## 2. Resumen estadístico de longitudes de episodio
Calculamos y mostramos medias, mínimos, máximos y desviaciones estándar por grid, agente y semilla.

In [2]:
# Resumen estadístico por grid, agente y semilla
pd.set_option('display.float_format', lambda x: '%.2f' % x)
if not df.empty:
    print('Columnas disponibles:', df.columns.tolist())
    print(df.head(10))
else:
    print('El DataFrame está vacío. Verifica la celda de carga de datos.')

Columnas disponibles: ['grid', 'seed', 'agent', 'mean', 'std', 'min', 'max']
   grid  seed        agent  mean  std  min  max
0     8    42      control  1.00 0.00    1    1
1     8    42    simbiosis  1.00 0.00    1    1
2     8    42  dqn_control  1.00 0.00    1    1
3     8   101      control  1.00 0.00    1    1
4     8   101    simbiosis  1.00 0.00    1    1
5     8   101  dqn_control  1.00 0.00    1    1
6     8    13      control  1.00 0.00    1    1
7     8    13    simbiosis  1.00 0.00    1    1
8     8    13  dqn_control  1.00 0.00    1    1
9     8     7      control  1.00 0.00    1    1


# Análisis científico de resultados F1 (protocolo v11)

Experimento de alto riesgo (risk_scale=1.5, risk_level=high) en grids 8×8 y 16×16.


## 1. Importar librerías y definir rutas


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

raw_dir = Path('../raw')
files = {
    'grid8': {
        'json': raw_dir / 'grid8_riskhigh_seed42_v11.json',
        'csv': raw_dir / 'grid8_riskhigh_seed42_v11_episodes.csv'
    },
    'grid16': {
        'json': raw_dir / 'grid16_riskhigh_seed42_v11.json',
        'csv': raw_dir / 'grid16_riskhigh_seed42_v11_episodes.csv'
    }
}
display(files)


{'grid8': {'json': WindowsPath('../raw/grid8_riskhigh_seed42_v11.json'),
  'csv': WindowsPath('../raw/grid8_riskhigh_seed42_v11_episodes.csv')},
 'grid16': {'json': WindowsPath('../raw/grid16_riskhigh_seed42_v11.json'),
  'csv': WindowsPath('../raw/grid16_riskhigh_seed42_v11_episodes.csv')}}

## 2. Cargar datos


In [12]:
# Cargar CSV
df_grid8 = pd.read_csv(files['grid8']['csv']) if files['grid8']['csv'].exists() else pd.DataFrame()
df_grid16 = pd.read_csv(files['grid16']['csv']) if files['grid16']['csv'].exists() else pd.DataFrame()
display(df_grid8.head())
display(df_grid16.head())

# Cargar JSON
def load_json(path):
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

data_grid8 = load_json(files['grid8']['json'])
data_grid16 = load_json(files['grid16']['json'])
list(data_grid8.keys())


""


""


[]

## 3. Limpieza básica


In [5]:
def clean_df(df, label):
    if df.empty:
        print(f"{label}: DataFrame vacío")
        return df
    print(f"{label}: shape={df.shape}")
    return df.dropna()

df_grid8_clean = clean_df(df_grid8, 'grid8')
df_grid16_clean = clean_df(df_grid16, 'grid16')


grid8: DataFrame vacío
grid16: DataFrame vacío


## 4. Visualización de métricas clave


In [6]:
metrics = ['Recompensa', 'Flexibilidad', 'Robustez', 'RiskEffective_Avg', 'Surprise_Avg', 'PGF_Bruto_Avg', 'PGF_Costo_Avg']

def plot_metric(df, metric, grid_label):
    if df.empty or metric not in df.columns:
        print(f"{grid_label}: métrica {metric} no disponible")
        return
    plt.figure(figsize=(8,5))
    sns.boxplot(x='Agente', y=metric, data=df)
    plt.title(f'{metric} por agente ({grid_label})')
    plt.show()

for metric in metrics:
    plot_metric(df_grid8_clean, metric, 'grid8')
    plot_metric(df_grid16_clean, metric, 'grid16')


grid8: métrica Recompensa no disponible
grid16: métrica Recompensa no disponible
grid8: métrica Flexibilidad no disponible
grid16: métrica Flexibilidad no disponible
grid8: métrica Robustez no disponible
grid16: métrica Robustez no disponible
grid8: métrica RiskEffective_Avg no disponible
grid16: métrica RiskEffective_Avg no disponible
grid8: métrica Surprise_Avg no disponible
grid16: métrica Surprise_Avg no disponible
grid8: métrica PGF_Bruto_Avg no disponible
grid16: métrica PGF_Bruto_Avg no disponible
grid8: métrica PGF_Costo_Avg no disponible
grid16: métrica PGF_Costo_Avg no disponible


## 5. Resumen estadístico por agente


In [7]:
def resumen_agente(df, label):
    if df.empty:
        print(f"{label}: sin datos")
        return
    print(f"--- {label} ---")
    print(df.groupby('Agente').agg({
        'Recompensa': ['mean', 'std'],
        'Flexibilidad': 'mean',
        'Robustez': 'mean',
        'RiskEffective_Avg': 'mean',
        'Surprise_Avg': 'mean',
        'PGF_Bruto_Avg': 'mean',
        'PGF_Costo_Avg': 'mean'
    }))

resumen_agente(df_grid8_clean, 'grid8')
resumen_agente(df_grid16_clean, 'grid16')


grid8: sin datos
grid16: sin datos


## 6. Exportar tabla resumen


In [8]:
summary_rows = []
for df, label in [(df_grid8_clean, 'grid8'), (df_grid16_clean, 'grid16')]:
    if df.empty:
        continue
    agg = df.groupby('Agente').agg({
        'Recompensa': ['mean', 'std'],
        'Flexibilidad': 'mean',
        'Robustez': 'mean',
        'RiskEffective_Avg': 'mean',
        'Surprise_Avg': 'mean',
        'PGF_Bruto_Avg': 'mean',
        'PGF_Costo_Avg': 'mean',
    })
    agg.columns = ['_'.join([str(c) for c in cols if c]) for cols in agg.columns.values]
    agg = agg.reset_index()
    agg.insert(0, 'Grid', label)
    summary_rows.append(agg)

if summary_rows:
    summary_df = pd.concat(summary_rows, ignore_index=True)
    out_path = Path('resumen_F1_v11_metricas.csv')
    summary_df.to_csv(out_path, index=False)
    print('Tabla resumen guardada en:', out_path)
    display(summary_df)
else:
    print('No hay datos para exportar.')


No hay datos para exportar.


## 7. Estad?stica inferencial (IC, p-valores, tama?os de efecto)

In [9]:
from pathlib import Path
from scipy import stats
from scipy.stats import mannwhitneyu
import pandas as pd
import numpy as np

# Configuraci?n de seeds/grids y rutas
seeds = [42, 101, 13, 7, 99]
grids = [8, 16]
metrics = ['Recompensa', 'Flexibilidad', 'Robustez', 'RiskEffective_Avg', 'Surprise_Avg', 'PGF_Bruto_Avg', 'PGF_Costo_Avg', 'IPG']

def load_episodes(grid, seed):
    path = Path(f"results/v11/F1_highrisk/raw/grid{grid}_riskhigh_r1p2_seed{seed}_v11_episodes.csv")
    if not path.exists():
        print(f"No encontrado: {path}")
        return None
    df = pd.read_csv(path)
    df['Grid'] = grid
    df['Seed'] = seed
    return df

# Cargar todos los episodios
dfs = []
for g in grids:
    for s in seeds:
        df_tmp = load_episodes(g, s)
        if df_tmp is not None:
            dfs.append(df_tmp)

df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"Datasets cargados: {len(dfs)}")
display(df_all.head())

No encontrado: results\v11\F1_highrisk\raw\grid8_riskhigh_r1p2_seed42_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid8_riskhigh_r1p2_seed101_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid8_riskhigh_r1p2_seed13_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid8_riskhigh_r1p2_seed7_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid8_riskhigh_r1p2_seed99_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid16_riskhigh_r1p2_seed42_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid16_riskhigh_r1p2_seed101_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid16_riskhigh_r1p2_seed13_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid16_riskhigh_r1p2_seed7_v11_episodes.csv
No encontrado: results\v11\F1_highrisk\raw\grid16_riskhigh_r1p2_seed99_v11_episodes.csv
Datasets cargados: 0


""


In [10]:
from scipy import stats
from scipy.stats import mannwhitneyu

def statistical_comparison(df, metric, agent1='simbiosis', agent2='control', alpha=0.05):
    data1 = df[df['Agente'] == agent1][metric].dropna()
    data2 = df[df['Agente'] == agent2][metric].dropna()
    if len(data1) == 0 or len(data2) == 0:
        return None

    # Normalidad
    p_norm1 = stats.shapiro(data1)[1] if len(data1) >= 3 else 0
    p_norm2 = stats.shapiro(data2)[1] if len(data2) >= 3 else 0

    if p_norm1 > 0.05 and p_norm2 > 0.05:
        test_used = "Welch's t-test"
        t_stat, p_val = stats.ttest_ind(data1, data2, equal_var=False)
    else:
        test_used = "Mann-Whitney U"
        u_stat, p_val = mannwhitneyu(data1, data2, alternative='two-sided')

    mean_diff = data1.mean() - data2.mean()
    pooled_std = np.sqrt((data1.std(ddof=1)**2 + data2.std(ddof=1)**2) / 2)
    cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
    if abs(cohens_d) < 0.2:
        effect = 'negligible'
    elif abs(cohens_d) < 0.5:
        effect = 'small'
    elif abs(cohens_d) < 0.8:
        effect = 'medium'
    else:
        effect = 'large'

    ci1 = stats.t.interval(0.95, len(data1)-1, loc=data1.mean(), scale=stats.sem(data1)) if len(data1) > 1 else (np.nan, np.nan)
    ci2 = stats.t.interval(0.95, len(data2)-1, loc=data2.mean(), scale=stats.sem(data2)) if len(data2) > 1 else (np.nan, np.nan)

    return {
        'metric': metric,
        'agent1': agent1, 'agent2': agent2,
        'test_used': test_used,
        'mean1': data1.mean(), 'std1': data1.std(ddof=1), 'ci1': ci1,
        'mean2': data2.mean(), 'std2': data2.std(ddof=1), 'ci2': ci2,
        'p_value': p_val, 'significant': p_val < alpha,
        'cohens_d': cohens_d, 'effect_size': effect
    }

results = []
if not df_all.empty:
    for g in grids:
        df_g = df_all[df_all['Grid'] == g]
        for metric in metrics:
            if metric in df_g.columns:
                r = statistical_comparison(df_g, metric)
                if r:
                    r['grid'] = g
                    results.append(r)

    alpha_corr = 0.05 / max(1, len(metrics))
    results_df = pd.DataFrame(results)
    if not results_df.empty:
        display(results_df.head())
        out_path = Path('analysis/stat_tests_F1_v11.csv')
        results_df.to_csv(out_path, index=False)
        print('Tabla de tests guardada en:', out_path)
        sig = results_df[results_df['p_value'] < alpha_corr]
        print(f'Significativos con Bonferroni (?_corr={alpha_corr:.4f}):')
        display(sig[['grid','metric','p_value','cohens_d','effect_size']])
    else:
        print('No hay resultados: faltan m?tricas en los CSV.')
else:
    print('df_all vac?o: revisa rutas/archivos.')

df_all vac?o: revisa rutas/archivos.


## 8. Exportar figuras/tablas clave

In [11]:
# Exportar longitudes por agente/grid/seed (del resumen de pgf_evol)
if 'df' in globals() and not df.empty:
    lengths_out = Path('analysis/longitudes_F1_v11.csv')
    df.to_csv(lengths_out, index=False)
    print('Tabla de longitudes guardada en', lengths_out)
